# 2.04 值迭代与策略迭代

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/02_rl/2.04_value_policy_iteration.ipynb)


## 为什么要学

上一课从"解方程"的角度得到了值迭代。这一课换个角度，从"**评估 → 改进**"的循环来理解求最优策略。这个循环叫**广义策略迭代（GPI）**，是几乎所有 RL 算法的共同骨架：

```text
     ┌──────── 策略评估：算出当前策略有多好（v 或 q）────────┐
     │                                                       ▼
  策略 π                                                   值 v_π
     ▲                                                       │
     └──────── 策略改进：对 q 取贪心（或往 q 大的方向走）◀────┘
```

- Actor-Critic：critic 做"评估"，actor 做"改进"；
- PPO：采样估计优势函数是"评估"，裁剪的策略梯度更新是"改进"；
- RLT：Q 网络是"评估"，actor 往 Q 增大的方向更新是"改进"。

对应书中第 4 章。

---

## 1. 值迭代（Value Iteration）

每次迭代两步：

1. **策略更新**：$\pi_{k+1}(s) = \arg\max_a\left[R(s,a) + \gamma\sum_{s'}p(s'|s,a)v_k(s')\right]$
2. **值更新**：$v_{k+1}(s) = \max_a\left[R(s,a) + \gamma\sum_{s'}p(s'|s,a)v_k(s')\right]$

注意 $v_k$ **不是**任何策略的真实值，它只是一个中间量。

## 2. 策略迭代（Policy Iteration）

每次迭代两步：

1. **策略评估**（PE）：**精确**求解 $\mathbf{v}_{\pi_k} = \mathbf{r}_{\pi_k}+\gamma P_{\pi_k}\mathbf{v}_{\pi_k}$（闭式解或迭代到收敛）；
2. **策略改进**（PI）：$\pi_{k+1}(s)=\arg\max_a q_{\pi_k}(s,a)$。

**策略改进定理**保证 $v_{\pi_{k+1}}\ge v_{\pi_k}$：每一轮都不会变差。有限 MDP 的确定性策略个数有限，所以一定在有限轮内收敛到最优策略。通常只需要很少几轮。

## 3. 截断策略迭代（Truncated Policy Iteration）

值迭代和策略迭代其实是同一个算法的两个极端：区别只在于**策略评估做多少步迭代**。

| 算法 | 策略评估迭代次数 $j$ |
|---|---|
| 值迭代 | $j = 1$ |
| 截断策略迭代 | $1 < j < \infty$ |
| 策略迭代 | $j = \infty$（精确求解） |

评估做得越精确，外层需要的轮数越少，但每轮更贵。实际中的深度 RL 算法几乎都是"截断"的：critic 只训练几步，就去更新 actor。

---

## 4. 代码

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vla_ladder.envs import GridWorld, policy_to_matrix

env = GridWorld()
gamma = 0.9

def policy_model(env, pi_det):
    """确定性策略 (S,) -> r_pi (S,), P_pi (S, S)"""
    pi = policy_to_matrix(pi_det, env.n_actions)
    return (pi * env.R).sum(1), np.einsum("sa,sat->st", pi, env.P)

def greedy(env, v, gamma):
    """对 v 取贪心：返回 (q (S,A), 贪心策略 (S,))"""
    q = env.R + gamma * env.P @ v
    return q, q.argmax(1)

### 练习 1：策略迭代

In [ ]:
def policy_iteration(env, gamma, max_iter=100):
    """返回 (v*, π*, 每轮的策略列表)"""
    pi = np.zeros(env.n_states, dtype=int)     # 初始策略：全部"向上"
    history = [pi.copy()]
    for _ in range(max_iter):
        # TODO:
        # 1) 策略评估：用闭式解求 v_pi
        # 2) 策略改进：pi_new = greedy(env, v_pi, gamma)[1]
        # 3) 如果 pi_new 与 pi 完全相同，说明收敛，break
        raise NotImplementedError
    return v, pi, history

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def policy_iteration(env, gamma, max_iter=100):
    """返回 (v*, π*, 每轮的策略列表)"""
    pi = np.zeros(env.n_states, dtype=int)
    history = [pi.copy()]
    for _ in range(max_iter):
        r_pi, P_pi = policy_model(env, pi)
        v = np.linalg.solve(np.eye(env.n_states) - gamma * P_pi, r_pi)   # 精确的策略评估
        _, pi_new = greedy(env, v, gamma)                                 # 策略改进
        if np.array_equal(pi_new, pi):
            break
        pi = pi_new
        history.append(pi.copy())
    return v, pi, history

In [ ]:
# ✅ 自测
v_pi, pi_pi, hist = policy_iteration(env, gamma)
print(f"策略迭代 {len(hist)} 轮收敛")
q, _ = greedy(env, v_pi, gamma)
assert np.allclose(v_pi, q.max(1), atol=1e-8)       # 满足贝尔曼最优方程
print("通过 ✅")
fig, axes = plt.subplots(1, min(4, len(hist)), figsize=(4 * min(4, len(hist)), 4))
for k, ax in enumerate(np.atleast_1d(axes)):
    r_pi, P_pi = policy_model(env, hist[k])
    env.plot(V=np.linalg.solve(np.eye(25) - gamma * P_pi, r_pi), policy=hist[k], title=f"PI iteration {k}", ax=ax)
plt.tight_layout(); plt.show()

观察图中策略的演化：**离目标近的状态先变好，然后"好消息"一圈圈地向外传播**。

### 练习 2：截断策略迭代

把"精确的策略评估"换成"从当前 $v$ 出发，做 $j$ 步迭代 $v\leftarrow r_\pi+\gamma P_\pi v$"。

In [ ]:
def truncated_policy_iteration(env, gamma, j, tol=1e-6, max_iter=10000):
    """返回 (v, π, 每轮的 ||v - v*||_∞ 误差列表)；需要用到全局变量 v_pi（真实最优值）来记录误差"""
    v = np.zeros(env.n_states)
    errors = []
    for _ in range(max_iter):
        # TODO:
        # 1) 策略改进：pi = greedy(env, v, gamma)[1]
        # 2) 截断的策略评估：对 pi 做 j 次 v = r_pi + γ P_pi v
        # 3) 记录 errors.append(np.max(np.abs(v - v_pi)))，若 < tol 则 break
        raise NotImplementedError
    return v, pi, errors

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def truncated_policy_iteration(env, gamma, j, tol=1e-6, max_iter=10000):
    """返回 (v, π, 每轮的 ||v - v*||_∞ 误差列表)；需要用到全局变量 v_pi（真实最优值）来记录误差"""
    v = np.zeros(env.n_states)
    errors = []
    for _ in range(max_iter):
        _, pi = greedy(env, v, gamma)
        r_pi, P_pi = policy_model(env, pi)
        for _ in range(j):
            v = r_pi + gamma * P_pi @ v
        errors.append(np.max(np.abs(v - v_pi)))
        if errors[-1] < tol:
            break
    return v, pi, errors

In [ ]:
# ✅ 自测 + 对比：j=1 就是值迭代
plt.figure(figsize=(5, 3.5))
for j in [1, 3, 10, 50]:
    v, pi, errs = truncated_policy_iteration(env, gamma, j)
    assert np.allclose(v, v_pi, atol=1e-5)
    plt.semilogy(errs, label=f"j={j}" + (" (= value iteration)" if j == 1 else ""))
    print(f"j={j:2d}: 外层 {len(errs):3d} 轮，总共 {len(errs) * j:4d} 次'评估迭代'")
plt.xlabel("outer iteration"); plt.ylabel("||v - v*||"); plt.legend(); plt.title("truncated policy iteration"); plt.show()
print("通过 ✅")

---

## 5. 三种算法的对比

| | 值迭代 | 截断策略迭代 | 策略迭代 |
|---|---|---|---|
| 每轮评估 | 1 步 | $j$ 步 | 精确求解 |
| 外层轮数 | 多 | 中 | 少 |
| 中间的 $v$ 是某个策略的真实值吗？ | 否 | 否 | 是 |
| 深度 RL 中的对应 | DQN（每步都对 max 做 TD 更新） | Actor-Critic、PPO、SAC | 很少用（太贵） |

## 小结

- **广义策略迭代（GPI）**：评估与改进交替进行，是所有 RL 算法的骨架。
- 策略迭代每轮都保证策略不变差，通常几轮就收敛。
- 值迭代与策略迭代是截断策略迭代的两个极端。

到这里，**有模型**的部分结束。但现实中我们不知道 $p(s'|s,a)$，因为你没法写出"线头插入"的物理模型。下一课开始学**无模型**方法：只靠与环境交互的数据来学习。